In [1]:
import numpy as np
from prompt_toolkit.styles import Attrs

from load_mtx import load_mtx

A, b = load_mtx("tasks/1_1.txt")
n = len(b)

A, b

(array([[ 1., -5., -7.,  1.],
        [ 1., -3., -9., -4.],
        [-2.,  4.,  2.,  1.],
        [-9.,  9.,  5.,  3.]]),
 array([-75., -41.,  18.,  29.]))

\begin{aligned}
Ax &= b, \\
P^{-1}LU &= A.
\end{aligned}

In [2]:
# Permutations для строк в матрице
P = np.eye(n, dtype=float)

L = np.eye(n, dtype=float)
U = A.copy()
swap_count = 0

for col in range(n):
    pivot_row = col + np.argmax(np.abs(U[col:, col]))

    if np.isclose(U[pivot_row, col], 0.0):
        raise ValueError("Матрица вырожденная: LU-разложение невозможно")

    if pivot_row != col:
        swap_count += 1
        U[[col, pivot_row], :] = U[[pivot_row, col], :]
        P[[col, pivot_row], :] = P[[pivot_row, col], :]

        L[[col, pivot_row], :col] = L[[pivot_row, col], :col]

    for row in range(col + 1, n):
        factor = U[row, col] / U[col, col]
        L[row, col] = factor
        U[row, col:] -= factor * U[col, col:]

U, L

(array([[-9.        ,  9.        ,  5.        ,  3.        ],
        [ 0.        , -4.        , -6.44444444,  1.33333333],
        [ 0.        ,  0.        , -5.22222222, -4.33333333],
        [ 0.        ,  0.        ,  0.        ,  2.93617021]]),
 array([[ 1.        ,  0.        ,  0.        ,  0.        ],
        [-0.11111111,  1.        ,  0.        ,  0.        ],
        [-0.11111111,  0.5       ,  1.        ,  0.        ],
        [ 0.22222222, -0.5       ,  0.44680851,  1.        ]]))

\begin{aligned}
Ux &= z, \\
Lz &= Pb.
\end{aligned}

In [3]:
from IPython.display import display, Math

pb = P @ b

z = np.zeros(n, dtype=float)
for row in range(n):
    z[row] = pb[row] - np.dot(L[row, :row], z[:row])

x = np.zeros(n, dtype=float)
for row in range(n - 1, -1, -1):
    s = np.dot(U[row, row + 1:], x[row + 1:])
    x[row] = (z[row] - s) / U[row, row]

for i in range(n):
    display(Math(fr"x_{i+1} = {x[i]:.3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
detA = (-1) ** swap_count * np.prod(np.diag(U))
display(Math(fr"det A = {detA:.3f}"))

<IPython.core.display.Math object>

\begin{aligned}
PA &= LU, \\
PAA^{-1} &= LUA^{-1}, \\
P &= LUA^{-1}.
\end{aligned}

Обозначим
\begin{aligned}
C &= UA^{-1}, \\
LC &= P, \\
UA^{-1} &= C.
\end{aligned}

In [14]:
A_inv = np.zeros((n, n), dtype=np.float64)

C = np.zeros((n, n), dtype=np.float64)
for row in range(n):
    C[row, :] = P[row, :] - L[row, :row] @ C[:row, :]

for row in range(n - 1, -1, -1):
    A_inv[row, :] = (C[row, :] - U[row, row + 1:] @ A_inv[row + 1:, :]) / U[row, row]

A_inv

array([[ 0.02898551, -0.0326087 ,  0.52536232, -0.22826087],
       [ 0.00724638,  0.05434783,  0.56884058, -0.11956522],
       [-0.10869565, -0.06521739, -0.2826087 ,  0.04347826],
       [ 0.24637681, -0.15217391,  0.34057971, -0.06521739]])

Проверка решения

In [5]:
np.allclose(x, np.linalg.solve(A, b))

True

Проверка корректности матриц L U и перестановки строк

In [6]:
np.allclose(A, P.T @ L @ U)

True

Проверка корректности вычисленного определителя

In [13]:
np.isclose(np.linalg.det(A), detA)

np.True_

Проверка корректности обратной матрицы

In [15]:
np.allclose(np.linalg.inv(A), A_inv)

True